# Week 4 — Retrieval-Augmented Generation (RAG)

**AI Agentic Engineering · Corte 1**

Companion notebook to `week-04-rag-content.html`. This is the last notebook before your Corte 1 project delivery
in Week 5 — the pipeline you build here is a working draft of that project.

**You will practice:**
1. Chunking a document with overlap.
2. Indexing chunks in ChromaDB, and the same vectors in FAISS for comparison.
3. A complete `answer_with_rag` function.
4. A light ADK agent with a `retrieve_context` tool, and the LangChain equivalent.
5. Two open exercises — including running this on your **own** project documents.


In [ ]:
%pip install -q --upgrade google-genai google-adk langchain-google-genai langgraph python-dotenv numpy chromadb faiss-cpu

In [1]:
import os
import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
MODEL = "gemini-flash-latest"
EMBED_MODEL = "gemini-embedding-001"

def embed(text):
    return client.models.embed_content(model=EMBED_MODEL, contents=text).embeddings[0].values

## 0. Sample corpus

A small self-contained knowledge base about this course, so the notebook runs without any external files. In the
lab, swap this for **your own project documents** (see Exercise 1).

In [2]:
COURSE_DOCS = """
AI Agentic Engineering is a 16-week elective course for Systems Engineering students at Universidad de
Santander. It is organized into three graded cuts called cortes. Corte 1 (weeks 1-5) covers LLM fundamentals,
context engineering, and RAG. Corte 2 (weeks 6-11) covers agents, multi-agent systems, Google ADK, and
LangGraph. Corte 3 (weeks 12-16) covers evaluation, observability, and deployment to production.

Corte 1 is worth 30% of the final grade: 25% for a practical project and 5% for in-class activities such as
quizzes, workshops, and labs. The Corte 1 project requires building a conversational assistant that combines
context engineering and RAG over a set of documents chosen by the student, using the Gemini API and ChromaDB,
delivered as a GitHub repository with a live 10-minute demonstration.

Corte 2 is also worth 30%: 25% for a multi-agent system project and 5% for in-class activities. Students must
build a multi-agent system that solves a real problem using Google ADK or LangGraph, integrating RAG
capabilities, delivered with a system diagram and a 15-minute live demonstration.

Corte 3 is worth 40% of the grade: 30% for a final integrator project and 10% for in-class activities. The
final project must combine agents, RAG, an automatic evaluation pipeline, tracing with Langfuse, and a REST
API exposed with FastAPI, delivered with a 5-minute demo video and a 20-minute technical presentation.

All labs in this course use free-tier tools: the Gemini API through Google AI Studio, ChromaDB and FAISS for
vector storage, and Langfuse's free tier for observability starting in Corte 3. Students are also introduced,
at a basic level, to Google Antigravity CLI, a terminal-based coding agent that can scaffold and edit project
files.

Class time each week is split into two blocks: two hours of theory with instructor-led demonstrations and
class discussion, followed by four hours of hands-on lab. Labs mix guided coding, small-group workshops,
individual work, and peer code review. All lab work is submitted through the course GitHub repository, with
Moodle used for supporting material and announcements.
"""

## 1. Chunking

In [3]:
def chunk_text(text: str, chunk_size: int = 400, overlap: int = 60) -> list[str]:
    text = " ".join(text.split())  # normalize whitespace
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

chunks = chunk_text(COURSE_DOCS, chunk_size=400, overlap=60)
print(f"{len(COURSE_DOCS)} chars -> {len(chunks)} chunks")
for i, c in enumerate(chunks):
    print(f"[{i}] {c[:70]}...")

2137 chars -> 7 chunks
[0] AI Agentic Engineering is a 16-week elective course for Systems Engine...
[1] s 12-16) covers evaluation, observability, and deployment to productio...
[2] s chosen by the student, using the Gemini API and ChromaDB, delivered ...
[3]  capabilities, delivered with a system diagram and a 15-minute live de...
[4]  5-minute demo video and a 20-minute technical presentation. All labs ...
[5] d coding agent that can scaffold and edit project files. Class time ea...
[6]  the course GitHub repository, with Moodle used for supporting materia...


## 2. Indexing with ChromaDB

In [4]:
import chromadb

chroma_client = chromadb.PersistentClient(path="./chroma_db_week4")
# start fresh each run of this notebook
try:
    chroma_client.delete_collection("course_docs")
except Exception:
    pass
collection = chroma_client.create_collection(name="course_docs")

collection.add(
    documents=chunks,
    embeddings=[embed(c) for c in chunks],
    ids=[f"chunk-{i}" for i in range(len(chunks))],
)
print(collection.count(), "chunks indexed")

7 chunks indexed


### 2a. The same vectors in FAISS (for comparison)

In [5]:
import faiss

vectors = np.array([embed(c) for c in chunks], dtype="float32")
faiss_index = faiss.IndexFlatL2(vectors.shape[1])
faiss_index.add(vectors)

query_vector = np.array([embed("How is Corte 1 graded?")], dtype="float32")
distances, indices = faiss_index.search(query_vector, k=3)
for dist, idx in zip(distances[0], indices[0]):
    print(f"{dist:.3f}  {chunks[idx][:80]}...")

0.684  s 12-16) covers evaluation, observability, and deployment to production. Corte 1...
0.716   capabilities, delivered with a system diagram and a 15-minute live demonstratio...
0.739  s chosen by the student, using the Gemini API and ChromaDB, delivered as a GitHu...


## 3. A complete `answer_with_rag` function

In [6]:
def answer_with_rag(question: str, n_results: int = 3, verbose: bool = False) -> str:
    results = collection.query(query_embeddings=[embed(question)], n_results=n_results)
    retrieved_chunks = results["documents"][0]
    if verbose:
        print("--- retrieved chunks ---")
        for c in retrieved_chunks:
            print("-", c[:80], "...")
    context = "\n\n---\n\n".join(retrieved_chunks)

    prompt = f'''Answer the question using ONLY the context below. If the answer isn't
in the context, say you don't have that information — do not make anything up.

Context:
{context}

Question: {question}
Answer:'''

    response = client.models.generate_content(
        model=MODEL, contents=prompt,
        config=types.GenerateContentConfig(temperature=0.1),
    )
    return response.text

print(answer_with_rag("How is Corte 1 graded?", verbose=True))
print()
print(answer_with_rag("What programming language does the course use?"))  # not in the docs -> should decline

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


--- retrieved chunks ---
- s 12-16) covers evaluation, observability, and deployment to production. Corte 1 ...
-  capabilities, delivered with a system diagram and a 15-minute live demonstratio ...
- s chosen by the student, using the Gemini API and ChromaDB, delivered as a GitHu ...
Based on the provided context, Corte 1 is worth 30% of the final grade, broken down as follows:
* **25%** for a practical project
* **5%** for in-class activities (such as quizzes, workshops, and labs)

I don't have that information.


## 4. Light preview: RAG as an ADK tool, and the LangChain equivalent

We're one step away from a real agent (Week 6 onward covers tool use and ReAct properly). For now, notice that
"retrieval" is just a Python function — which is exactly what an ADK **tool** is.

In [7]:
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner

def retrieve_context(query: str) -> dict:
    """Retrieve the most relevant course-document chunks for a query."""
    results = collection.query(query_embeddings=[embed(query)], n_results=3)
    return {"chunks": results["documents"][0]}

rag_agent = Agent(
    model=MODEL,
    name="course_rag_agent",
    instruction=(
        "Use the retrieve_context tool to find relevant chunks before answering. "
        "Answer ONLY using information returned by the tool. If it's not there, say so."
    ),
    tools=[retrieve_context],
)

async def ask_adk_agent(agent, prompt, app_name="week4_app", user_id="student"):
    runner = InMemoryRunner(agent=agent, app_name=app_name)
    session = await runner.session_service.create_session(app_name=app_name, user_id=user_id)
    content = types.Content(role="user", parts=[types.Part.from_text(text=prompt)])
    final_text = None
    async for event in runner.run_async(user_id=user_id, session_id=session.id, new_message=content):
        if event.content and event.content.parts and event.content.parts[0].text:
            final_text = event.content.parts[0].text
    return final_text

print(await ask_adk_agent(rag_agent, "How is Corte 2 graded?"))

c:\Users\José Martínez\Desktop\cosas de la U\ai_agentic\.venv\Lib\site-packages\google\adk\models\llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


**Corte 2** is worth **30%** of the final grade and is broken down as follows:

* **25% – Multi-Agent System Project:** Students must build a multi-agent system that solves a real problem using Google ADK or LangGraph, integrating RAG capabilities. It is delivered with a system diagram and a 15-minute live demonstration.
* **5% – In-Class Activities:** Quizzes, workshops, and labs.


In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model=MODEL)

def answer_with_rag_langchain(question: str, n_results: int = 3) -> str:
    results = collection.query(query_embeddings=[embed(question)], n_results=n_results)
    context = "\n\n---\n\n".join(results["documents"][0])
    prompt = (
        "Answer the question using ONLY the context below. If the answer isn't in the "
        "context, say you don't have that information.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )
    return llm.invoke(prompt).content

print(answer_with_rag_langchain("What tools are used for observability in this course?"))

[{'type': 'text', 'text': "Based on the provided context, Langfuse's free tier is used for observability.", 'extras': {'signature': 'EoEFCv4EARFNMg/9AOg4nIDNewgXdHk526nJ3e0HfVnBgYV5QxUmDzlJXFzXbshy8tvO2kMJL1oZusuq2cge1mYgAONox1qtG8Pz5EYDgNc+QDRPYUsUJsYnSYrwLyvXAjbMvL7kNZ+jaYoZ33UGzY1ZkwR917LT0cEbqSy04TId7jwWvDSHrD0QQeAh90u1HD1AmwPIZhMJyDFb48W4xADsOh+tf6RR1R6NqKxztzT4iNjTlnkCw/TjozhLBNAoxEXVwprCmim8tv7KvNeIotGmuV54IL95DCgIIspRFDpptJVnXO3EXH+d/dXQo2ZSmicUkKui7YuUzWhnKPNvj1EAkHAssOgA9dl4rvig+TN9VSDodx+gNbH9AbJ9AyMVYbdBpS6eWQkLWYZkrFj9zzsa3CCgTKzuDcuFcozWWUimXrFUANO8ioD16kXSNO9/ozRGmJyfGbomH8ReXME8Vshn256wzKaGHXsqWP/Sz3bHbST5TRnWXf8mOYEDPQH/FpTg7cOZ3CF3nyFYTGwCmcSIHwXtBmh2AhkFEPKZb7OxzcnXQRswK+C/4xM3ka9EcEjfrr9n6IALOHsVZn0SW53hIGXh6TnPM4rauYz4P5149L9UmFuWoSd6TMXzQs9BWpZhOIFStlpg9MTJzKheSyK+s59PRPKm8c1rcwHzxNA4w5ik9Xlupwht2nAwGDZa0Ytc60cZHTIuZM89x/lUbQwTSlT5KJLz0o3JejkBGEz2/DziNAdjo1BVkwX9KxhgricN/RHlDTHdBWX2gi5VWK76dLEH/hOcWYxFVicSu5DCy+XyCCmARBrL9RVNfPyGkTNss4osDxzLxA5relIZocw8L2c='}}]


## 5. Exercises

In [9]:
# Exercise 1 — Run this on YOUR OWN Corte 1 project documents
# Project: calculus tutoring assistant. data/ has 3 short placeholder notes (limits.txt,
# derivatives.txt, integrals.txt) standing in for the real lecture notes/textbook chapters
# (gathering the real materials is still pending from the Week 3 checkpoint).

import glob

DATA_DIR = "data"
doc_paths = sorted(glob.glob(os.path.join(DATA_DIR, "*.txt")))
print("Files found:", [os.path.basename(p) for p in doc_paths])

calc_chunks = []
calc_sources = []
for path in doc_paths:
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()
    file_chunks = chunk_text(text, chunk_size=400, overlap=60)
    calc_chunks.extend(file_chunks)
    calc_sources.extend([os.path.basename(path)] * len(file_chunks))

print(f"{len(calc_chunks)} chunks total from {len(doc_paths)} files")

calc_chroma_client = chromadb.PersistentClient(path="./chroma_db_week4_calc")
try:
    calc_chroma_client.delete_collection("calculus_docs")
except Exception:
    pass
calc_collection = calc_chroma_client.create_collection(name="calculus_docs")

calc_collection.add(
    documents=calc_chunks,
    embeddings=[embed(c) for c in calc_chunks],
    ids=[f"chunk-{i}" for i in range(len(calc_chunks))],
    metadatas=[{"source": s} for s in calc_sources],
)
print(calc_collection.count(), "calculus chunks indexed")

def answer_calc_rag(question, n_results=3, where=None):
    kwargs = {"query_embeddings": [embed(question)], "n_results": n_results}
    if where:
        kwargs["where"] = where
    results = calc_collection.query(**kwargs)
    context = "\n\n---\n\n".join(results["documents"][0])
    prompt = f'''Answer the question using ONLY the context below. If the answer isn't
in the context, say you don't have that information — do not make anything up.

Context:
{context}

Question: {question}
Answer:'''
    response = client.models.generate_content(
        model=MODEL, contents=prompt,
        config=types.GenerateContentConfig(temperature=0.1),
    )
    return response.text

questions = [
    "What is the chain rule and when do I use it?",
    "Why do we always add + C when finding an indefinite integral?",
    "What's the difference between a limit existing and a function being continuous at that point?",
    "How does the Fundamental Theorem of Calculus connect derivatives and integrals?",
    "When should I use integration by parts instead of substitution?",
]
for q in questions:
    print(f"\nQ: {q}")
    print("A:", answer_calc_rag(q))

Files found: ['derivatives.txt', 'integrals.txt', 'limits.txt']
13 chunks total from 3 files
13 calculus chunks indexed

Q: What is the chain rule and when do I use it?
A: Based on the provided context:

* **What it is:** If $f(x) = g(h(x))$, then $f'(x) = g'(h(x)) * h'(x)$.
* **When to use it:** It is used when differentiating composite functions (functions with an "outer" and "inner" function).

Q: Why do we always add + C when finding an indefinite integral?
A: Based on the provided context, we always include "+ C" because two antiderivatives of the same function can only differ by a constant (representing the entire family of antiderivatives of the function).

Q: What's the difference between a limit existing and a function being continuous at that point?
A: Based on the provided context, the difference is:

* **A limit existing:** The two-sided limit of a function $f(x)$ as $x$ approaches $c$ exists if both one-sided limits (the left-hand limit and the right-hand limit) exist and 

In [10]:
# Exercise 2 — Metadata filtering
# calc_collection was already indexed with metadatas=[{"source": ...}] in Exercise 1.

print("--- query with no filter (searches all 3 documents) ---")
results = calc_collection.query(query_embeddings=[embed("rate of change")], n_results=3)
for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print(f"[{meta['source']}] {doc[:70]}...")

print("\n--- same query, filtered to source=integrals.txt only ---")
results = calc_collection.query(
    query_embeddings=[embed("rate of change")],
    n_results=3,
    where={"source": "integrals.txt"},
)
for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print(f"[{meta['source']}] {doc[:70]}...")

print("\nFiltered answer using only integrals.txt:")
print(answer_calc_rag("What does this document say about rate of change?", where={"source": "integrals.txt"}))

--- query with no filter (searches all 3 documents) ---
[derivatives.txt] Derivatives The derivative of a function f at a point x measures the i...
[integrals.txt] te the volume of a solid of revolution, the total displacement of an o...
[integrals.txt] onstant. The definite integral of f(x) from a to b, written as the int...

--- same query, filtered to source=integrals.txt only ---
[integrals.txt] te the volume of a solid of revolution, the total displacement of an o...
[integrals.txt] onstant. The definite integral of f(x) from a to b, written as the int...
[integrals.txt] ls: it states that if F is an antiderivative of f, then the definite i...

Filtered answer using only integrals.txt:
I don't have that information.


## Corte 1 wrap-up

You now have every piece needed for your Week 5 project: a system prompt (Week 2), context/history handling
(Week 3), and a working RAG pipeline (this week). See `week-04-rag-activities.html` for the full delivery
checklist and grading rubric.

## Looking ahead

Corte 2 starts in Week 6 with agent fundamentals — tool use, function calling, and the ReAct pattern — building
directly on the `tools=[retrieve_context]` preview above.